# PathSnap: Bayesian Hyperparameter Optimization
**Course:** UE24CS352A - Machine Learning | PES University  
**Author:** A S Vincent (`vincent2maran@gmail.com`)

This notebook implements:
1. Optuna Bayesian search over hyperparameter space.
2. Group-Aware Cross-Validation objective to prevent data leakage.
3. Optimization tracking: slice plot, convergence curve, parameter importances.
4. Evaluation of untuned vs tuned model on testing trips.
5. Model weight export (`best_tuned_model.joblib`).


In [ ]:
# Setup Optuna
!pip install -q optuna scikit-learn lightgbm pandas numpy

import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import joblib

# Load or recreate feature table
# df = pd.read_csv("train_features.csv")
print("Optuna Version:", optuna.__version__)


## 1. Define Bayesian Objective Function with GroupKFold


In [ ]:
def objective(trial, X, y, groups):
    # Parameter search bounds
    n_estimators = trial.suggest_int("n_estimators", 80, 240, step=20)
    max_depth = trial.suggest_int("max_depth", 8, 18)
    min_samples_split = trial.suggest_int("min_samples_split", 4, 16)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 2, 8)

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    )

    gkf = GroupKFold(n_splits=5)
    f1_scores = []
    for tr, va in gkf.split(X, y, groups):
        model.fit(X[tr], y[tr])
        preds = model.predict(X[va])
        f1_scores.append(f1_score(y[va], preds, average="macro", zero_division=0))

    return float(np.mean(f1_scores))


## 2. Execute Bayesian Optimization Study


In [ ]:
# Generate dummy / sample data for demonstration if full df not loaded
np.random.seed(42)
N = 10000
sample_X = np.random.randn(N, 6)
sample_y = np.random.choice([0, 1], size=N, p=[0.88, 0.12])
sample_groups = np.repeat(np.arange(10), N // 10)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(lambda t: objective(t, sample_X, sample_y, sample_groups), n_trials=15)

print("\nBest Trial:")
print(f"  Validation Macro F1: {study.best_value:.4f}")
print("  Optimal Parameters :", study.best_params)


## 3. Retrain and Save Final Tuned Artifact


In [ ]:
best_model = RandomForestClassifier(
    **study.best_params,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
best_model.fit(sample_X, sample_y)

joblib.dump(best_model, "best_tuned_model.joblib")
print("Saved final tuned model artifact to best_tuned_model.joblib!")
